# RoadSense AI: Multi-Task Machine Learning for Accident Severity and Casualties
## Notebook 02: Exploratory Data Analysis & Statistical Risk Insights
**Course Module**: Artificial Intelligence (CO5: Classification, Regression and Prediction Problems)
**Target Team Member**: Member 2 (Exploratory Data Analysis & Visualizations Lead)

---

### Objectives:
1. Explore class distributions for Task A (Binary) and Task B (3-class) to quantify severe class imbalance.
2. Analyze bivariate relationship between posted speed limit and danger rate (Fatal + Serious injury probability).
3. Examine environmental risk factors: weather conditions, road geometry, and ambient lighting.
4. Analyze physiological vulnerability of road users (pedal cycles, motorcycles vs passenger cars).
5. Identify temporal risk spikes across 24-hour diurnal cycle.
6. Inspect casualty count distribution and heavy-tailed extreme outliers (70 casualties bus collision).
7. Save publication-quality figures to `../outputs/charts/`.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath(".."))
from src.eda import (
    plot_class_distributions,
    plot_danger_by_speed_limit,
    plot_danger_by_weather,
    plot_danger_by_road_type,
    plot_danger_by_vehicle_type,
    plot_danger_by_hour,
    plot_danger_by_light_condition,
    plot_casualties_distribution
)
print("Imports loaded successfully.")

Imports loaded successfully.


## 1. Load Clean Master Dataset
Loading the multi-year dataset (2021-2023) created in Stage 1.

In [2]:
df = pd.read_parquet("../data/clean_roadsense.parquet")
print(f"Loaded master clean dataset: {len(df):,} records across {len(df.columns)} columns.")
df.head(3)

Loaded master clean dataset: 311,349 records across 29 columns.


,collision_index,collision_year,police_force,target_severity_binary,target_severity_3class,target_casualties,speed_limit,weather_conditions,road_type,light_conditions,...,month,number_of_vehicles,any_motorcycle,any_pedal_cycle,any_hgv_or_bus,most_vulnerable_vehicle_type,age_of_vehicle_max,age_of_vehicle_max_missing,driver_age_band_youngest,driver_age_band_youngest_missing
0,2021170M10151,2021,17.0,0,2,1,30,1.0,6.0,1.0,...,2,1,0,0,0,car_taxi,NaN,1,9.0,0
1,2021170S11701,2021,17.0,1,1,1,30,1.0,3.0,4.0,...,10,2,0,1,0,pedal_cycle,NaN,1,3.0,0
2,2021111064046,2021,11.0,0,2,1,30,1.0,6.0,1.0,...,7,2,0,0,0,car_taxi,10.0,0,6.0,0


## 2. Target Class Distributions & Severe Imbalance
- **Task A**: Minor (~76.7%) vs Dangerous (~23.3%)
- **Task B**: Slight (~76.7%) vs Serious (~21.9%) vs Fatal (~1.48%)
The severe scarcity of Fatal accidents demands specialized reweighting, SMOTE, and PR-AUC tracking in subsequent stages.

In [3]:
plot_class_distributions(df, save_path="../outputs/charts/01_class_distribution.png")
print("Task A Counts:\n", df["target_severity_binary"].value_counts(normalize=True)*100)
print("Task B Counts:\n", df["target_severity_3class"].value_counts(normalize=True)*100)

[+] Saved: ../outputs/charts/01_class_distribution.png
Task A Counts:
 target_severity_binary
0    76.665093
1    23.334907
Name: proportion, dtype: float64
Task B Counts:
 target_severity_3class
2    76.665093
1    21.858108
0     1.476799
Name: proportion, dtype: float64


## 3. Danger Rate vs. Posted Speed Limit
Because kinetic energy scales quadratically ($E_k = \frac{1}{2}mv^2$), collisions on high-speed roads (60-70 mph) yield significantly higher danger rates (>35%) and fatal rates compared to urban 20-30 mph zones (<20%).

In [4]:
plot_danger_by_speed_limit(df, save_path="../outputs/charts/02_danger_rate_by_speed_limit.png")

[+] Saved: ../outputs/charts/02_danger_rate_by_speed_limit.png


## 4. Environmental Risk: Weather & Road Surface
Analyzing risk rates across adverse weather conditions (Fog, High Winds, Heavy Rain).

In [5]:
plot_danger_by_weather(df, save_path="../outputs/charts/03_danger_rate_by_weather.png")

[+] Saved: ../outputs/charts/03_danger_rate_by_weather.png


## 5. Road Geometry: Junctions & Carriageway Types
Roundabouts show the lowest danger rate despite frequent minor collisions, due to forced low-speed angle mergers.

In [6]:
plot_danger_by_road_type(df, save_path="../outputs/charts/04_danger_rate_by_road_type.png")

[+] Saved: ../outputs/charts/04_danger_rate_by_road_type.png


## 6. Vehicle Vulnerability Hierarchy
Vulnerable road users (cyclists and motorcyclists) lack protective metal cages, crumple zones, and airbags. Motorcyclists face an extreme danger rate exceeding 40%.

In [7]:
plot_danger_by_vehicle_type(df, save_path="../outputs/charts/05_danger_rate_by_vehicle_type.png")

[+] Saved: ../outputs/charts/05_danger_rate_by_vehicle_type.png


## 7. Diurnal Rhythm: Volume vs Lethality
Peak accident volume occurs during evening rush hour (16:00 - 18:00), but **peak danger rate occurs in early morning hours (01:00 - 04:00)** due to higher speeds, reduced visibility, and driver fatigue.

In [8]:
plot_danger_by_hour(df, save_path="../outputs/charts/06_danger_rate_by_hour.png")

[+] Saved: ../outputs/charts/06_danger_rate_by_hour.png


## 8. Environmental Lighting
Darkness without street lighting presents the highest risk of serious and fatal injuries.

In [9]:
plot_danger_by_light_condition(df, save_path="../outputs/charts/07_danger_rate_by_light_condition.png")

[+] Saved: ../outputs/charts/07_danger_rate_by_light_condition.png


## 9. Casualty Count Distribution & Extreme Outliers
- Over 80.6% of collisions result in exactly 1 casualty.
- The 2023 dataset includes an extreme 70-casualty bus crash.
- As instructed, this outlier is preserved in the test set to evaluate Poisson loss and robust metrics (Median AE alongside MAE and RMSE).

In [10]:
plot_casualties_distribution(df, save_path="../outputs/charts/08_casualties_distribution_and_outlier.png")
print("Casualties summary statistics:")
print(df["target_casualties"].describe(percentiles=[0.5, 0.75, 0.9, 0.99]))

[+] Saved: ../outputs/charts/08_casualties_distribution_and_outlier.png
Casualties summary statistics:
count    311349.000000
mean          1.274024
std           0.704422
min           1.000000
50%           1.000000
75%           1.000000
90%           2.000000
99%           4.000000
max          70.000000
Name: target_casualties, dtype: float64
